# 🥇 From 14th Place to 0.8+: A Leakage-Safe Thermal Baseline for CUHK-X HAR

I am currently **14th on the leaderboard with a score above 0.80**, within the Finalist race. This notebook shares the deliberately pre-optimization version of the approach: Thermal frames only, subject-wise validation, uniform temporal sampling, and clip-level logit averaging. It is designed to expose useful failure modes and leave room for experimentation.

**Important:** never use a random clip split for this task. Clips from the same person are highly correlated.

In [1]:
from pathlib import Path
import random, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 2026
NUM_CLASSES, NUM_FRAMES, IMAGE_SIZE = 40, 8, 112
EPOCHS, BATCH_SIZE, NUM_WORKERS = 8, 32, 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('device:', DEVICE)

device: cuda


In [2]:
# Find the attached competition data without assuming Kaggle's slug/path.
search_roots = [Path('/kaggle/input'), Path('data/raw/extracted')]
thermal_dirs = []
for root in search_roots:
    if not root.exists(): continue
    # Avoid rglob over the full competition archive; inspect only common layouts.
    probes = [root/'Thermal', root/'HAR'/'data'/'Thermal', root/'data'/'Thermal']
    for child in root.iterdir():
        if child.is_dir(): probes += [child/'Thermal', child/'HAR'/'data'/'Thermal', child/'data'/'Thermal']
    thermal_dirs += [p for p in probes if p.is_dir()]
if not thermal_dirs:
    DATA_ROOT = None
    print('Thermal data was not attached. Attach the CUHK-X competition data, then rerun the data cells.')
else:
    DATA_ROOT = next((p.parent for p in thermal_dirs if (p.parent/'Thermal').is_dir()), thermal_dirs[0].parent)
    print('DATA_ROOT:', DATA_ROOT)

Thermal data was not attached. Attach the CUHK-X competition data, then rerun the data cells.


In [3]:
IMAGE_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
def natural(p):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)', p.name)]
def discover(root):
    rows = []
    for action in sorted((root/'Thermal').iterdir()):
        m = re.match(r'^(\d+)', action.name)
        if not m: continue
        for user in sorted(action.glob('user*')):
            um = re.match(r'user(\d+)', user.name, re.I)
            if not um: continue
            for trial in sorted(p for p in user.iterdir() if p.is_dir()):
                frames = sorted([p for p in trial.rglob('*') if p.suffix.lower() in IMAGE_EXT], key=natural)
                if frames: rows.append(dict(label=int(m.group(1)), user=int(um.group(1)), frames=frames, clip=str(trial)))
    return pd.DataFrame(rows)
df = discover(DATA_ROOT) if DATA_ROOT is not None else pd.DataFrame(columns=['label','user','frames','clip'])
if len(df):
    assert set(df.user).issubset(set(range(1,10)) | set(range(16,25))), 'Unexpected user: possible leakage.'
    gkf = GroupKFold(5); df['fold'] = -1
    for fold, (_, va) in enumerate(gkf.split(df, df.label, df.user)): df.loc[va, 'fold'] = fold
    print(df.shape, 'clips'); display(df.groupby('fold').agg(clips=('label','size'), users=('user','nunique'))); display(df.groupby('label').size().describe())
else: print('No clips available in this runtime; the notebook code is ready once the competition data is attached.')

No clips available in this runtime; the notebook code is ready once the competition data is attached.


In [4]:
def sample_indices(n, k, train):
    edges = np.linspace(0, n, k+1); out=[]
    for a,b in zip(edges[:-1], edges[1:]):
        lo, hi = min(int(a), n-1), min(max(int(np.ceil(b))-1, int(a)), n-1)
        out.append(random.randint(lo,hi) if train else (lo+hi)//2)
    return out
class ThermalClips(Dataset):
    def __init__(self, table, train=False): self.t, self.train = table.reset_index(drop=True), train
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        row=self.t.iloc[i]; fs=row.frames; imgs=[]
        flip=self.train and random.random()<.5
        for j in sample_indices(len(fs), NUM_FRAMES, self.train):
            im=Image.open(fs[j]).convert('RGB').resize((IMAGE_SIZE,IMAGE_SIZE))
            if flip: im=im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            x=torch.from_numpy(np.asarray(im, dtype=np.float32)).permute(2,0,1)/255.0
            imgs.append((x-.5)/.25)
        return torch.stack(imgs), int(row.label), i

In [5]:
class Block(nn.Module):
    def __init__(self, a,b,s=1):
        super().__init__(); self.c=nn.Sequential(nn.Conv2d(a,b,3,s,1,bias=False),nn.BatchNorm2d(b),nn.ReLU(),nn.Conv2d(b,b,3,1,1,bias=False),nn.BatchNorm2d(b)); self.d=nn.Sequential(nn.Conv2d(a,b,1,s,bias=False),nn.BatchNorm2d(b)) if (a!=b or s!=1) else nn.Identity()
    def forward(self,x): return torch.relu(self.c(x)+self.d(x))
class FrameNet(nn.Module):
    def __init__(self):
        super().__init__(); self.f=nn.Sequential(nn.Conv2d(3,32,7,2,3,bias=False),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(3,2,1),Block(32,32),Block(32,64,2),Block(64,128,2),Block(128,256,2),nn.AdaptiveAvgPool2d(1)); self.fc=nn.Linear(256,NUM_CLASSES)
    def forward(self,x):
        b,t,c,h,w=x.shape; z=self.f(x.reshape(b*t,c,h,w)).flatten(1); return self.fc(z).reshape(b,t,-1).mean(1)
def loaders(train, valid):
    return (DataLoader(ThermalClips(train,True),BATCH_SIZE,True,num_workers=NUM_WORKERS,pin_memory=True), DataLoader(ThermalClips(valid),BATCH_SIZE,False,num_workers=NUM_WORKERS,pin_memory=True))

In [6]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); pred=[]; truth=[]
    for x,y,_ in loader: pred.append(model(x.to(DEVICE)).argmax(1).cpu()); truth.append(y)
    pred,truth=torch.cat(pred),torch.cat(truth); return (pred==truth).float().mean().item(), pred, truth
def train_fold(fold=0):
    tr,va=df[df.fold!=fold],df[df.fold==fold]; tl,vl=loaders(tr,va); model=FrameNet().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4); loss_fn=nn.CrossEntropyLoss()
    for ep in range(EPOCHS):
        model.train(); total=0
        for x,y,_ in tl: opt.zero_grad(); loss=loss_fn(model(x.to(DEVICE)),y.to(DEVICE)); loss.backward(); opt.step(); total+=loss.item()
        acc,_,_=evaluate(model,vl); print(f'fold {fold} epoch {ep+1:02d}: loss={total/len(tl):.3f} val_acc={acc:.3f}')
    return model, va, vl
if len(df): model, valid_df, valid_loader = train_fold(0)
else: model = valid_df = valid_loader = None

In [7]:
if model is not None:
    acc, pred, truth = evaluate(model, valid_loader)
    print('held-out subject accuracy:', round(acc,4))
    fig, ax = plt.subplots(figsize=(10,10)); ConfusionMatrixDisplay(confusion_matrix(truth, pred, labels=range(NUM_CLASSES))).plot(ax=ax, xticks_rotation='vertical', colorbar=False); plt.show()
else: print('Attach the competition data to run training and evaluation.')
# Suggested experiments: change NUM_FRAMES, replace mean pooling, add temporal GRU, or compare folds.
# Keep every comparison subject-wise and report the split explicitly.

Attach the competition data to run training and evaluation.
